# Notebook A — Feature‑based Liveness Evaluation (TFLite)
# -----------------------------------------------------
# Evaluates: liveness_model.tflite + liveness_scaler.pkl
# Dataset format (CSV):
# session_id,yaw,pitch,roll,motion,label
# label: 1=live, 0=spoof
# -----------------------------------------------------

In [2]:
import sys
print (sys.executable)

c:\Users\Leo\Documents\AI\Capstone\.venv\Scripts\python.exe


In [1]:
!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.7 MB 9.6 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.7 MB 10.6 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.7 MB 11.0 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.7 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 11.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from collections import deque

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd()) # put notebook in app/notebooks/
MODELS_DIR = os.path.join(BASE_DIR, "models")
DATA_DIR = os.path.join(BASE_DIR, "feature_dataset")


MODEL_PATH = os.path.join(MODELS_DIR, "liveness_model.tflite")
SCALER_PATH = os.path.join(MODELS_DIR, "liveness_scaler.pkl")
DATASET_CSV = os.path.join(DATA_DIR, "features.csv")


WINDOW_SIZE = 16


print("BASE_DIR:", BASE_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("DATASET:", DATASET_CSV)

In [ ]:
scaler = joblib.load(SCALER_PATH)


interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()


input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


print("Model loaded")

In [ ]:
df = pd.read_csv(DATASET_CSV)
print(df.head())

In [ ]:
def build_windows(df, window_size=16):
X, y = [], []
for session_id, group in df.groupby("session_id"):
group = group.sort_index()
features = group[["yaw", "pitch", "roll", "motion"]].values
labels = group["label"].values


for i in range(len(features) - window_size + 1):
window = features[i:i+window_size]
label = labels[i+window_size-1]
X.append(window)
y.append(label)


return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


X_raw, y_true = build_windows(df, WINDOW_SIZE)
print("Windows:", X_raw.shape)

In [ ]:
flat = X_raw.reshape(-1, 4)
flat = scaler.transform(flat)
X_norm = flat.reshape(len(X_raw), WINDOW_SIZE, 4).astype(np.float32)

In [ ]:
def predict_tflite(X):
preds = []
for i in range(len(X)):
sample = X[i:i+1]
interpreter.set_tensor(input_details[0]['index'], sample)
interpreter.invoke()
score = interpreter.get_tensor(output_details[0]['index'])[0][0]
preds.append(score)
return np.array(preds)


scores = predict_tflite(X_norm)
y_pred = (scores >= 0.5).astype(int)

In [ ]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)


print("Accuracy :", round(acc,4))
print("Precision:", round(prec,4))
print("Recall :", round(rec,4))
print("Confusion Matrix:\n", cm)

In [ ]:
results = pd.DataFrame({
"accuracy": [acc],
"precision": [prec],
"recall": [rec]
})


results_path = os.path.join(DATA_DIR, "feature_model_results.csv")
results.to_csv(results_path, index=False)
print("Saved to", results_path)